In [69]:
import os
import xml.etree.ElementTree as ET
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
import torchvision.transforms as transforms

In [70]:
class PedestrianDataset(Dataset):
    def __init__(self, frames_dir, labels_dir, transform=None):
        self.frames_dir = frames_dir
        self.labels_dir = labels_dir
        self.transform = transform
        
        # Match PNGs with their corresponding XML files dynamically
        self.filenames = []
        for f in sorted(os.listdir(frames_dir)):
            if f.endswith('.png'):
                base_name = os.path.splitext(f)[0]
                xml_name = f"{base_name}.xml"
                if os.path.exists(os.path.join(labels_dir, xml_name)):
                    self.filenames.append(base_name)
                    
        print(f"Matched {len(self.filenames)} image/label pairs successfully.")

    def __len__(self):
        return len(self.filenames)

    def parse_xml(self, xml_path, idx):
        tree = ET.parse(xml_path)
        root = tree.getroot()
        
        boxes = []
        labels = []
        areas = []
        
        for obj in root.findall('object'):
            name = obj.find('name').text
            
            # CRITICAL: Match 'person' from your XML file
            if name == 'person':
                labels.append(1)  # Class 1 = Person (Class 0 is background)
                
                # Extract Bounding Box
                bndbox = obj.find('bndbox')
                xmin = float(bndbox.find('xmin').text)
                ymin = float(bndbox.find('ymin').text)
                xmax = float(bndbox.find('xmax').text)
                ymax = float(bndbox.find('ymax').text)
                
                boxes.append([xmin, ymin, xmax, ymax])
                
                # Calculate box area (width * height)
                area = (xmax - xmin) * (ymax - ymin)
                areas.append(area)
        
        # Convert everything to standard PyTorch Tensors
        target = {}
        target["boxes"] = torch.as_tensor(boxes, dtype=torch.float32)
        target["labels"] = torch.as_tensor(labels, dtype=torch.int64)
        target["image_id"] = torch.tensor([idx], dtype=torch.int64)
        target["area"] = torch.as_tensor(areas, dtype=torch.float32)
        target["iscrowd"] = torch.zeros((len(labels),), dtype=torch.int64) # Assuming no crowd tags
        
        return target

    def __getitem__(self, idx):
        base_name = self.filenames[idx]
        
        img_path = os.path.join(self.frames_dir, f"{base_name}.png")
        xml_path = os.path.join(self.labels_dir, f"{base_name}.xml")
        
        # Your XML lists depth=1 (Grayscale). 
        # Note: Most torchvision detection models still expect 3 channels (RGB).
        # Convert to "RGB" (replicates the grayscale channel 3 times) to avoid model errors.
        image = Image.open(img_path).convert("RGB") 
        
        target = self.parse_xml(xml_path, idx)
        
        if self.transform:
            image = self.transform(image)
            
        return image, target

In [71]:
# Define the absolute or relative paths to your folders
TRAIN_DIR = "data/train" 
FRAMES_DIR = os.path.join(TRAIN_DIR, "Pedestrian frame")
LABELS_DIR = os.path.join(TRAIN_DIR, "Pedestrian label") 

In [72]:
import torch
import torch.nn as nn
import snntorch as snn
from snntorch import surrogate

In [73]:
class SpikingPedestrianDetector(nn.Module):
    def __init__(self, beta=0.9, num_steps=3):
        super().__init__()
        self.num_steps = num_steps # Number of time steps to simulate spikes
        
        # Spike gradient surrogate function (allows backpropagation over discrete spikes)
        spike_grad = surrogate.fast_sigmoid(slope=25)
        
        # 1. Spiking Convolutional Backbone
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1) # Downsample 256 -> 128
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1) # Downsample 128 -> 64
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1) # Downsample 64 -> 32
        self.lif3 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        
        # 2. Dense Predictor Heads (Reads flattened spike features)
        # 64 channels * 32 * 32 spatial grid = 65,536 features
        self.flatten = nn.Flatten()
        
        # Bounding Box Regressor (predicts 4 coordinates: xmin, ymin, xmax, ymax)
        self.bbox_head = nn.Linear(64 * 32 * 32, 4) 
        
        # Classification Head (predicts probability of 'person' presence)
        self.cls_head = nn.Linear(64 * 32 * 32, 1) 

    def forward(self, x):
        # Initialize membrane potentials for LIF neurons
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        
        # Accumulators for spikes/outputs over time steps
        spk3_sum = torch.zeros((x.size(0), 64, 32, 32), device=x.device)
        
        # Static input emulation: feed the same image across all time steps
        for step in range(self.num_steps):
            cur1 = self.conv1(x)
            spk1, mem1 = self.lif1(cur1, mem1)
            
            cur2 = self.conv2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)
            
            cur3 = self.conv3(spk2)
            spk3, mem3 = self.lif3(cur3, mem3)
            
            spk3_sum += spk3 # Accumulate spikes over time
            
        # Average spike feature map across time
        feat = spk3_sum / self.num_steps
        feat_flat = self.flatten(feat)
        
        # Final predictions
        pred_boxes = self.bbox_head(feat_flat)
        pred_logits = self.cls_head(feat_flat)
        
        return pred_boxes, pred_logits

In [ ]:
# 1. Transforms to handle 256x256 conversion
image_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

# 2. Instantiate Dataset and Loader
dataset = PedestrianDataset(FRAMES_DIR, LABELS_DIR, transform=image_transforms)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

generator = torch.Generator().manual_seed(42)
train_dataset, test_dataset = random_split(dataset, [train_size, test_size], generator=generator)

print(f"Total Images: {len(dataset)} | Training Split: {len(train_dataset)} | Testing Split: {len(test_dataset)}")

def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0, collate_fn=collate_fn)

# 3. Model, Optimizer, and Loss functions
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SpikingPedestrianDetector(beta=0.9, num_steps=5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Losses
bbox_loss_fn = nn.MSELoss() # bounding boxes
cls_loss_fn = nn.BCEWithLogitsLoss() # object classification

Matched 4119 image/label pairs successfully.
Total Images: 4119 | Training Split: 3295 | Testing Split: 824


In [75]:
def calculate_iou(boxA, boxB):
    """
    Calculates Intersection over Union (IoU) between two boxes.
    Boxes format: [xmin, ymin, xmax, ymax]
    """
    # Determine the coordinates of the intersection rectangle
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    # Compute area of intersection
    interArea = max(0, xB - xA) * max(0, yB - yA)

    # Compute area of both bounding boxes
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBAArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    # Compute Union area
    unionArea = boxAArea + boxBAArea - interArea

    if unionArea == 0:
        return 0.0
        
    return interArea / float(unionArea)

In [76]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    
    for images, targets in train_loader:
        # 1. Skip empty frames / unpack safely depending on batch layout
        # if targets is a single dict (batch_size=1), wrap it in a tuple to match batch logic
        if isinstance(targets, dict):
            targets = (targets,)
            
        # Check if the first image in the batch has a box (for this basic regression setup)
        if targets[0]["boxes"].size(0) == 0:
            continue 

        # 2. Process image tensor dimensions using the type check fix from earlier
        if isinstance(images, (tuple, list)):
            images = torch.stack(images).to(device)
        elif isinstance(images, torch.Tensor):
            if images.dim() == 3:
                images = images.unsqueeze(0) 
            images = images.to(device)

        # 3. Forward Pass through the Spiking Neural Network
        pred_boxes, pred_logits = model(images)
        
        # 4. Compute Loss across the batch
        # For this basic regression setup, we will gather targets from the batch items
        batch_loss = 0
        orig_w, orig_h = 346, 260 # Dimensions from your a130.xml
        
        for i in range(len(targets)):
            # --- THE FIX HAPPENS HERE ---
            # Look inside target 'i' of the tuple, then fetch the first box tensor
            if targets[i]["boxes"].size(0) == 0:
                continue
                
            true_box = targets[i]["boxes"][0].to(device) # Fetch first box coordinates
            
            # Normalize coordinates [0, 1]
            true_box_scaled = torch.tensor([
                true_box[0] / orig_w,
                true_box[1] / orig_h,
                true_box[2] / orig_w,
                true_box[3] / orig_h
            ], device=device)
            
            true_cls = torch.tensor([1.0], device=device)
            
            # Calculate loss against model outputs for index i
            loss_box = bbox_loss_fn(pred_boxes[i], true_box_scaled)
            loss_cls = cls_loss_fn(pred_logits[i], true_cls)
            
            batch_loss += (loss_box + loss_cls)
            
        # Average the loss over the batch size
        total_loss = batch_loss / len(targets)
        
        # 5. Optimization step
        optimizer.zero_grad()
        total_loss.backward()
        optimizer.step()
        
        epoch_loss += total_loss.item()
        
    print(f"Epoch [{epoch+1}/{num_epochs}] | Total Loss: {epoch_loss:.4f}")

Epoch [1/10] | Total Loss: 20.1330
Epoch [2/10] | Total Loss: 6.2703
Epoch [3/10] | Total Loss: 1.5381
Epoch [4/10] | Total Loss: 1.6636
Epoch [5/10] | Total Loss: 1.0142
Epoch [6/10] | Total Loss: 0.6729
Epoch [7/10] | Total Loss: 0.4802
Epoch [8/10] | Total Loss: 0.3726
Epoch [9/10] | Total Loss: 0.3444
Epoch [10/10] | Total Loss: 0.2635


In [77]:
def test_model_accuracy(model, data_loader, device):
    model.eval() # Turn off spike gradient surrogate noise and dropout
    
    total_samples = 0
    correct_classifications = 0
    total_iou = 0.0
    valid_boxes_count = 0
    
    orig_w, orig_h = 346, 260 # Dimensions from your dataset
    
    with torch.no_grad():
        for images, targets in data_loader:
            if isinstance(targets, dict):
                targets = (targets,)
                
            # Process and stack image batch tensor
            if isinstance(images, (tuple, list)):
                images = torch.stack(images).to(device)
            elif isinstance(images, torch.Tensor):
                if images.dim() == 3: images = images.unsqueeze(0)
                images = images.to(device)
                
            # Forward Pass through SNN
            pred_boxes, pred_logits = model(images)
            
            # Read predictions over the current batch
            for i in range(len(targets)):
                total_samples += 1
                
                # A. Classification Check (Threshold logits at 0.0 for binary BCE)
                has_person_prediction = (pred_logits[i].item() >= 0.0)
                actual_has_person = (targets[i]["boxes"].size(0) > 0)
                
                if has_person_prediction == actual_has_person:
                    correct_classifications += 1
                
                # B. Bounding Box Accuracy Check (Only if both have a box to compare)
                if actual_has_person:
                    true_box = targets[i]["boxes"][0].to(device)
                    
                    # Scale true box to [0,1] format to match network output scale
                    true_box_scaled = torch.tensor([
                        true_box[0] / orig_w,
                        true_box[1] / orig_h,
                        true_box[2] / orig_w,
                        true_box[3] / orig_h
                    ], device=device)
                    
                    # Calculate IoU score for this prediction
                    iou_score = calculate_iou(pred_boxes[i].cpu().numpy(), true_box_scaled.cpu().numpy())
                    total_iou += iou_score
                    valid_boxes_count += 1
                    
    # Calculate Final Metrics
    cls_accuracy = (correct_classifications / total_samples) * 100
    avg_iou = (total_iou / valid_boxes_count) * 100 if valid_boxes_count > 0 else 0.0
    
    print("\n================ TEST SPLIT RESULTS ================")
    print(f"Total Evaluated Images:       {total_samples}")
    print(f"Pedestrian Detection Accuracy: {cls_accuracy:.2f}%")
    print(f"Average Bounding Box IoU:     {avg_iou:.2f}%")
    print("====================================================")
    
    return cls_accuracy, avg_iou

In [78]:
test_model_accuracy(model, test_loader, device)


================ TEST SPLIT RESULTS ================
Total Evaluated Images:       824
Pedestrian Detection Accuracy: 93.20%
Average Bounding Box IoU:     15.32%


(93.20388349514563, np.float32(15.317387))